In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np

from robot_wm.inference.actor.base import RobotActionHistory, RobotObsHistory
from robot_wm.inference.actor.cost.visual_based_latents_cost import \
    L2VisualLatentsCost
from robot_wm.inference.robot.base import RobotObs
from robot_wm.inference.task.reference_episode import ImageProprioGoal
from robot_wm.utils.config import from_config

In [ ]:
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence

import hydra
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from IPython import display
from omegaconf import OmegaConf
from PIL import Image, ImageOps
from tqdm.notebook import tqdm, trange

# from robot_actions.utils.notebook import prepare_inputs, save_rollout, view_rollout
from robot_wm.utils.notebook import prepare_inputs, save_rollout, view_rollout

os.environ["COSMOS_HOME"] = "/fxs-cortex/carohiguera/vt_wm"

## Dataset

We start by loading the droid dataset

In [ ]:
dataset = from_config(
    "datasets/configs/datasets/droid.yaml",
    # can also use RAD_paths.csv for the heldout set.
    overrides=[
        "manifest=/fsx-cortex-datacache/shared/datasets/droid/011825/droid_h5/train_paths.csv"
    ],
)

MPK dataset without wrist camera

In [ ]:
dataset = from_config(
    "datasets/configs/datasets/mpk.yaml",
    overrides=[
        "manifest_patterns=['**/evaluation_tasks/**/folding/**/episode.h5', '**/evaluation_tasks/**/pick/**/episode.h5', '**/evaluation_tasks/**/push/**/episode.h5', '**/evaluation_tasks/**/text_goal/**/episode.h5']"
    ]
)

MPK dataset with wrist camera

In [ ]:
dataset = from_config(
    "datasets/configs/datasets/mpk.yaml",
    overrides=[
        "manifest_patterns=[**/evaluation_tasks/mpk/pick/reachliftcup_v1/run_0001/episode.h5,**/evaluation_tasks/mpk/pick/pickcube_v0/run_0001/episode.h5,**/evaluation_tasks/mpk/pick/pickpen_v0/run_0001/episode.h5,**/evaluation_tasks/mpk/push/brownboxpush_v0/run_0001/episode.h5,**/evaluation_tasks/mpk/folding/foldjacketsleeve_v1/run_0001/episode.h5]"
    ]
)

In [ ]:


def sample_to_obs_act_history(sample):
    
    obs = sample["episode_data"]["observation"]
    
    observation_history = RobotObsHistory(max_context=1000, freq=30)
    for i in range(len(obs["joint_position"])):
        joints = obs["joint_position"][i]
        
        ee_pose = obs["cartesian_position"][i]
        gripper = obs["gripper_position"][i]
        ee_pose_with_gripper = np.concatenate((ee_pose, np.array([gripper])))
        
        cam = obs["exterior_image_2_left"][i]
        cam = np.transpose(cam, (2, 0, 1)) / 255.0
        robot_obs = RobotObs(joints, ee_pose_with_gripper, cam)
        observation_history.push(robot_obs)
    
    action_history = observation_history.get_action_history_from_ee_deltas()
    return observation_history, action_history

len(dataset)

In [ ]:

sample = dataset[3]
def print_dict_structure(d, prefix=''):
    for k, v in d.items():
        full_key = f"{prefix}/{k}" if prefix else k
        if isinstance(v, dict):
            print(f"[Group]   {full_key}")
            print_dict_structure(v, full_key)
        elif hasattr(v, 'shape') and hasattr(v, 'dtype'):
            print(f"[Dataset] {full_key}: shape={v.shape}, dtype={v.dtype}")
        else:
            print(f"[Unknown] {full_key}: type={type(v)}")

print_dict_structure(sample)


In [ ]:
observation_history, action_history = sample_to_obs_act_history(dataset[3])
observation_history.show()

In [ ]:
len(observation_history.obs)

In [ ]:
action_history.actions[100].action

## World Model
Instantiate a world model from our model zoo or _menagerie_. 

> Each world model requires you to have setup their corresponding project, see corresponding README's or Cortex Wiki for more details.

In [ ]:
# can add overrides to load different weights or change params

wm = from_config('menagerie/config/st_wm.yaml')
# wm = from_config('menagerie/config/dino_wm.yaml')
# wm = from_config("menagerie/config/jepa_wm.yaml")

#### Does the world model encode correctly an observation stream?

We encode observations from droid, no rollouts. This should look very similar to the dataset video.

In [ ]:
# Some models don't allow a very large context for rollouts, you could see and change that parameter for this visualization at wm.max_context
wm.max_context = 1000
ans = wm.encode_history(observation_history, action_history)
imgs = wm.decode_latents(ans)
imgs.show()

#### Does the WM does sensible future predictions?
Now encode a small history and make predictions predictions using GT actions.

In [ ]:
init, context, end = 0, 20, -1

# Take 20 frames of context
context_obs = observation_history[init : init + context]
context_actions = context_obs.get_action_history_from_ee_deltas()

# Encode them
history = wm.encode_history(context_obs, context_actions)

# Take all the future GT actions
future_obs = observation_history[init + context :]  # can be used for comparison
future_actions = future_obs.get_action_history_from_ee_deltas()

# Rollout and show them
rollout = wm.rollout(history, future_actions)
pred_imgs = wm.decode_latents(rollout)
print(pred_imgs.obs[0].image.shape)   
pred_imgs.show()
# print(pred_imgs.obs[0].image.shape)
# future_obs.show()
# imgs.show()

In [ ]:
context_obs.obs[-1].end_effector_pose

## one step inference

In [ ]:
# get future action sequences
N = len(future_actions.actions)
future_action_sequences = torch.tensor([future_actions.actions[i].action for i in range(N)]).permute(1, 0, 2)


In [ ]:
future_action_sequences.shape

In [ ]:
one_step_action = future_action_sequences[:, 0:5, :].reshape(1,1,35)


new_context = history
for i in range(2):
    one_step_action = future_action_sequences[:, i*5:(i+1)*5, :].reshape(1,1,35)
    new_context, z_new_visual = wm._one_step_latent_pred(new_context, one_step_action)
rollout = wm.decode_latents(new_context)
rollout.show()



In [ ]:
rollout.obs[-1].image.shape

## WM inference server
- pip install Pyro5 (Pyro5 is a python version RPC)
- change the host with the current machine, ```daemon = Pyro5.api.Daemo(host="a100-st-p4de24xlarge-132")  ```
- copy the output URI to client

In [ ]:
import Pyro5.api
import numpy as np
from matplotlib import pyplot as plt
import cv2
import time
from scipy.spatial.transform import Rotation as R

# new_context = history
@Pyro5.api.expose
class ControllerInterface:
    def __init__(self):
        self.step_count = 0
        self.new_context = history

    def step(self, data_dict):  # data_dict: {'action': [...], 'step': int}
        action_chunk = data_dict["action"]
        
        action_chunk = torch.tensor(action_chunk[:5])  # Ensure action_chunk is a tensor
        # start_time = time.time()
        # action = action_chunk[0]
        # action = np.array(action) # delta eef
        # delta_eef_action = action[:-1] #* (1 / DROID_CONTROL_FREQUENCY_HZ)
        # gripper_state_action = action[-1]  # last element is gripper width
        # target_gripper_width = 1.0 - gripper_state_action  # convert to width


        one_step_action = action_chunk.reshape(1,1,35)
        self.new_context, z_new_visual = wm._one_step_latent_pred(self.new_context, one_step_action)
        rollout = wm.decode_latents(self.new_context)
        last_image = rollout.obs[-1].image
        last_image = (last_image * 255).astype(np.uint8)
        # print(last_image)
        # rollout.show()

        # print(f"control Step {self.step_count} | Received action: {action}")



        self.step_count += 1

        # output gif every 10 steps
        if self.step_count % 10 == 0:
            rollout.show()
        return {
            "left_image": last_image.tolist(),  # Convert to list for serialization
            "step": self.step_count
        }

# Pyro5 server
daemon = Pyro5.api.Daemon(host="a100-st-p4de24xlarge-132")
uri = daemon.register(ControllerInterface)
# ns.register("gr00t_WM_controller", uri)  # Register the object with the name server
print("Controller server running at:")
print(uri)
daemon.requestLoop()